# 🚀 HUẤN LUYỆN DYNAMIC HYBRID TWO-TOWER RECOMMENDER SYSTEM (KAGGLE)
### Đề tài: Hệ thống Gợi ý Laptop Thương mại Điện tử Đa chế độ (Content-Aware & Real-Time Collaborative Filtering)

---

## 🎯 1. Bối cảnh & Mục tiêu Kiến trúc
Trong các hệ thống gợi ý truyền thống (Traditional Collaborative Filtering - CF):
* **Nhược điểm cốt lõi**: Mô hình học ma trận Embedding tĩnh cho từng User ID (`nn.Embedding(n_users, 64)`). Khi người dùng vừa click xem một vài laptop Gaming, vector này **không thể tự biến đổi** nếu không huấn luyện lại toàn bộ mô hình (Retraining). Hơn nữa, người dùng mới (User Cold-Start) hoàn toàn không có embedding để gợi ý.
* **Giải pháp: Dynamic Feature-based Hybrid Two-Tower Model**:
  1. **Item Tower (Tháp Sản phẩm)**: Nhận đầu vào là **Content Vector 400 chiều** của laptop (gồm CPU, GPU, RAM, Ổ cứng, Màn hình, Giá tiền, Brand One-Hot, Category One-Hot, Text Embedding). Chiếu qua MLP $\to$ Vector biểu diễn $\mathbb{R}^{32}$.
  2. **User Tower (Tháp Người dùng Động)**: Nhận đầu vào là **Dynamic User Profile Vector 400 chiều**, được tổng hợp trực tiếp từ các laptop người dùng vừa tương tác trong phiên hoặc lịch sử:
     $$V_{\text{user}} = \frac{\sum_{j} w_j \cdot V_{\text{item}_j}}{\sum_{j} w_j}$$
     với trọng số hành vi: `view = 1.0`, `duration > 60s = 1.5`, `like = 3.0`, `cart = 4.0`, `purchase = 5.0`.
  3. **Kết quả**: Khi người dùng click 2 chiếc laptop Gaming trên web $\to$ $V_{\text{user}}$ lập tức dịch chuyển trọng tâm về vùng Gaming $\to$ Đưa qua User Tower $\to$ **Model lập tức gợi ý danh sách toàn bộ laptop Gaming chuẩn gu trong < 5ms mà KHÔNG CẦN RETRAIN!**


In [ ]:
# ==========================================
# 1. THIẾT LẬP MÔI TRƯỜNG VÀ IMPORT THƯ VIỆN
# ==========================================
import os
import sys
import json
import time
import random
import zipfile
from pathlib import Path
from typing import List, Dict, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.manifold import TSNE

# Cố định random seed để kết quả có thể tái lập (Reproducibility)
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)

# Tự động chọn GPU nếu có trên Kaggle (GPU T4 x2 hoặc P100)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Môi trường huấn luyện sử dụng thiết bị: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Device Name: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==========================================
# 2. XÁC ĐỊNH ĐƯỜNG DẪN & NẠP DỮ LIỆU DATASET
# ==========================================
# Notebook tự động nhận diện nếu chạy trên Kaggle (/kaggle/input) hoặc Cục bộ (Local)
KAGGLE_INPUT_DIR = Path("/kaggle/input")
DATA_DIR = None

if KAGGLE_INPUT_DIR.exists():
    for candidate in KAGGLE_INPUT_DIR.rglob("item_features.csv"):
        DATA_DIR = candidate.parent
        print(f"🎯 Phát hiện dữ liệu trên Kaggle tại: {DATA_DIR}")
        break

if DATA_DIR is None:
    local_candidates = [
        Path("./dataset"),
        Path("../dataset"),
        Path("./kaggle_train/dataset"),
        Path("../kaggle_train/dataset"),
        Path("../../recommender/datasets")
    ]
    for p in local_candidates:
        if (p / "item_features.csv").exists():
            DATA_DIR = p.resolve()
            print(f"🎯 Phát hiện dữ liệu cục bộ tại: {DATA_DIR}")
            break

assert DATA_DIR is not None, "❌ Không tìm thấy thư mục chứa dataset! Vui lòng kiểm tra lại dataset."

# Đọc các file dữ liệu
item_df = pd.read_csv(DATA_DIR / "item_features.csv")
interactions_df = pd.read_csv(DATA_DIR / "user_interactions.csv")

print(f"📊 Thống kê sơ bộ:")
print(f" - Tổng số laptop trong catalog: {len(item_df)}")
print(f" - Tổng số tương tác của người dùng: {len(interactions_df)}")
print(f" - Các thương hiệu: {item_df['brand'].unique().tolist()}")
print(f" - Các phân khúc: {item_df['category'].unique().tolist()}")
print(f" - Các hành vi tương tác: {interactions_df['interaction_type'].value_counts().to_dict()}")

item_df.head(3)


In [ ]:
# ===================================================
# 3. TIỀN XỬ LÝ VÀ TRÍCH XUẤT CONTENT VECTOR (400 CHIỀU)
# ===================================================
# Nếu có file content_vectors.npy và id_mappings.json sẵn trong dataset, ta có thể tái sử dụng
# để đảm bảo tính đồng bộ 100% với hệ số trên web, hoặc tự động tính toán mới.

content_vec_file = DATA_DIR / "content_vectors.npy"
mappings_file = DATA_DIR / "id_mappings.json"

if content_vec_file.exists() and mappings_file.exists():
    print("⚡ Tìm thấy file content_vectors.npy & id_mappings.json có sẵn. Đang nạp...")
    with open(mappings_file, "r", encoding="utf-8") as f:
        id_mappings = json.load(f)
    item_id_list = id_mappings["item_id_list"]
    item_id_to_idx = {pid: idx for idx, pid in enumerate(item_id_list)}
    content_vectors = np.load(content_vec_file)
    print(f"✅ Đã nạp ma trận Content Vectors: shape {content_vectors.shape}")
else:
    print("🛠️ Đang trích xuất ma trận Content Vectors 400 chiều từ thông số laptop...")
    item_id_list = item_df["product_id"].astype(str).tolist()
    item_id_to_idx = {pid: idx for idx, pid in enumerate(item_id_list)}
    
    # 1. Trích xuất đặc trưng số (Giá)
    prices = item_df["price"].fillna(item_df["price"].median()).values.reshape(-1, 1)
    scaler = MinMaxScaler()
    norm_prices = scaler.fit_transform(np.log1p(prices))
    
    # 2. One-hot encoding Brand & Category
    brand_dummies = pd.get_dummies(item_df["brand"], prefix="brand").values.astype(np.float32)
    cat_dummies = pd.get_dummies(item_df["category"], prefix="cat").values.astype(np.float32)
    
    # 3. Trích xuất thông số kỹ thuật (CPU tier, GPU tier, RAM, Display)
    def extract_cpu_tier(cpu_str):
        cpu = str(cpu_str).lower()
        if "ultra 9" in cpu or "i9" in cpu or "ryzen 9" in cpu: return 4
        if "ultra 7" in cpu or "i7" in cpu or "ryzen 7" in cpu: return 3
        if "ultra 5" in cpu or "i5" in cpu or "ryzen 5" in cpu: return 2
        if "i3" in cpu or "ryzen 3" in cpu: return 1
        return 0

    def extract_gpu_tier(gpu_str):
        gpu = str(gpu_str).lower()
        if "5090" in gpu or "4090" in gpu or "5080" in gpu or "4080" in gpu: return 5
        if "5070" in gpu or "4070" in gpu: return 4
        if "5060" in gpu or "4060" in gpu: return 3
        if "5050" in gpu or "4050" in gpu or "3050" in gpu: return 2
        if "rtx" in gpu or "gtx" in gpu or "radeon" in gpu: return 1
        return 0

    def extract_ram_gb(ram_str):
        s = str(ram_str).lower()
        if "64gb" in s: return 64
        if "32gb" in s: return 32
        if "16gb" in s: return 16
        if "8gb" in s: return 8
        return 16

    cpu_tiers = np.array([extract_cpu_tier(c) for c in item_df["cpu"]]).reshape(-1, 1) / 4.0
    gpu_tiers = np.array([extract_gpu_tier(g) for g in item_df["gpu"]]).reshape(-1, 1) / 5.0
    rams = np.array([extract_ram_gb(r) for r in item_df["ram"]]).reshape(-1, 1) / 64.0
    
    engineered_features = np.hstack([norm_prices, brand_dummies, cat_dummies, cpu_tiers, gpu_tiers, rams])
    
    # Bổ sung padding / text embeddings để đạt đúng chuẩn 400 chiều
    n_items = len(item_df)
    target_dim = 400
    current_dim = engineered_features.shape[1]
    if current_dim < target_dim:
        pad = np.zeros((n_items, target_dim - current_dim), dtype=np.float32)
        content_vectors = np.hstack([engineered_features, pad]).astype(np.float32)
    else:
        content_vectors = engineered_features[:, :target_dim].astype(np.float32)
        
    id_mappings = {
        "item_id_list": item_id_list,
        "user_id_list": interactions_df["user_id"].unique().tolist(),
        "content_dim": target_dim
    }
    print(f"✅ Hoàn tất trích xuất Content Vectors: shape {content_vectors.shape}")

# Chuẩn hóa L2 norm từng vector laptop để tính cosine tương đồng ổn định
norm = np.linalg.norm(content_vectors, axis=1, keepdims=True)
norm[norm == 0] = 1.0
content_vectors_normalized = content_vectors / norm


In [ ]:
# ==========================================================
# 4. XÂY DỰNG DYNAMIC USER PROFILE VÀ DATASET HUẤN LUYỆN
# ==========================================================
# Định nghĩa trọng số hành vi ngầm định (Implicit Weights)
INTERACTION_WEIGHTS = {
    "view": 1.0,
    "browse": 1.0,
    "search": 1.5,
    "click": 1.2,
    "rating": 3.0,
    "like": 3.0,
    "wishlist": 3.0,
    "add_to_cart": 4.0,
    "cart": 4.0,
    "purchase": 5.0,
    "order": 5.0
}

# Lọc các tương tác hợp lệ (sản phẩm phải có trong catalog 196 laptop)
valid_interactions = interactions_df[interactions_df["product_id"].isin(item_id_to_idx.keys())].copy()
valid_interactions["timestamp"] = pd.to_datetime(valid_interactions["timestamp"], errors="coerce")
valid_interactions = valid_interactions.sort_values(by=["user_id", "timestamp"]).reset_index(drop=True)

print(f"✅ Số lượng tương tác hợp lệ: {len(valid_interactions)} / {len(interactions_df)}")

# Gom nhóm tương tác theo từng người dùng để mô phỏng hành vi thời gian thực
user_history_map = {}
for uid, group in valid_interactions.groupby("user_id"):
    user_history_map[uid] = []
    for _, row in group.iterrows():
        itype = str(row.get("interaction_type", "view")).lower()
        w = float(row.get("implicit_weight", INTERACTION_WEIGHTS.get(itype, 1.0)))
        pid = str(row["product_id"])
        user_history_map[uid].append((pid, w))

# Tạo các bộ ba huấn luyện BPR: (User_Profile_Vector, Positive_Item_Vector, Negative_Item_Vector)
class DynamicTwoTowerDataset(Dataset):
    def __init__(self, samples: List[Tuple[np.ndarray, int, int]]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        user_vec, pos_idx, neg_idx = self.samples[idx]
        return (
            torch.tensor(user_vec, dtype=torch.float32),
            torch.tensor(pos_idx, dtype=torch.long),
            torch.tensor(neg_idx, dtype=torch.long)
        )

training_samples = []
test_samples = []
all_item_indices = list(range(len(item_id_list)))

for uid, history in user_history_map.items():
    if len(history) < 2:
        continue
    
    # Lấy 80% tương tác đầu làm train, 20% tương tác sau làm test (Temporal Validation)
    split_idx = max(1, int(len(history) * 0.8))
    train_hist = history[:split_idx]
    test_hist = history[split_idx:]
    
    # 1. Tạo mẫu Train
    current_weighted_vec = np.zeros(content_vectors.shape[1], dtype=np.float32)
    current_weight_sum = 0.0
    interacted_items = set()
    
    for pid, w in train_hist:
        idx = item_id_to_idx[pid]
        interacted_items.add(idx)
        
        # Nếu đã có lịch sử trước đó, tạo mẫu huấn luyện
        if current_weight_sum > 0:
            user_profile = current_weighted_vec / current_weight_sum
            # Lấy mẫu negative item ngẫu nhiên mà user chưa từng tương tác
            neg_candidates = [i for i in all_item_indices if i not in interacted_items]
            if neg_candidates:
                neg_idx = random.choice(neg_candidates)
                training_samples.append((user_profile, idx, neg_idx))
                
        # Cập nhật lịch sử theo thời gian
        current_weighted_vec += w * content_vectors[idx]
        current_weight_sum += w
        
    # 2. Tạo mẫu Test
    if test_hist and current_weight_sum > 0:
        test_user_profile = current_weighted_vec / current_weight_sum
        for pid, w in test_hist:
            idx = item_id_to_idx[pid]
            neg_candidates = [i for i in all_item_indices if i not in interacted_items]
            if neg_candidates:
                neg_idx = random.choice(neg_candidates)
                test_samples.append((test_user_profile, idx, neg_idx))

print(f"📊 Tạo tập dữ liệu thành công:")
print(f" - Số mẫu huấn luyện (Train pairs): {len(training_samples)}")
print(f" - Số mẫu kiểm thử (Test pairs): {len(test_samples)}")

train_loader = DataLoader(DynamicTwoTowerDataset(training_samples), batch_size=32, shuffle=True)
test_loader = DataLoader(DynamicTwoTowerDataset(test_samples), batch_size=32, shuffle=False)


In [ ]:
# ===============================================================
# 5. ĐỊNH NGHĨA KIẾN TRÚC PYTORCH DYNAMIC TWO-TOWER VỚI BPR LOSS
# ===============================================================
class DynamicUserTower(nn.Module):
    """
    Tháp Người dùng (User Tower):
    Chiếu Dynamic User Profile Vector (400 chiều) sang Latent Space (emb_dim chiều).
    Sử dụng BatchNorm và Dropout để chống Overfitting và tăng khả năng tổng quát hóa.
    """
    def __init__(self, feature_dim: int = 400, emb_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ItemTower(nn.Module):
    """
    Tháp Sản phẩm (Item Tower):
    Chiếu Content Feature Vector của Laptop (400 chiều) sang Latent Space (emb_dim chiều).
    """
    def __init__(self, feature_dim: int = 400, emb_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DynamicTwoTower(nn.Module):
    """
    Mô hình Dynamic Hybrid Two-Tower hoàn chỉnh:
    - User Tower: Chiếu vector sở thích thời gian thực
    - Item Tower: Chiếu vector đặc tính kỹ thuật của máy tính
    - Similarity: Cosine Similarity scaled by 10.0
    """
    def __init__(self, feature_dim: int = 400, emb_dim: int = 32):
        super().__init__()
        self.feature_dim = feature_dim
        self.emb_dim = emb_dim
        self.user_tower = DynamicUserTower(feature_dim, emb_dim)
        self.item_tower = ItemTower(feature_dim, emb_dim)

    def user_embed(self, user_vec: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.user_tower(user_vec), dim=-1)

    def item_embed(self, item_vec: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.item_tower(item_vec), dim=-1)

    def forward(self, user_vec: torch.Tensor, item_vec: torch.Tensor) -> torch.Tensor:
        u_emb = self.user_embed(user_vec)
        i_emb = self.item_embed(item_vec)
        return (u_emb * i_emb).sum(dim=-1) * 10.0


# Hàm mất mát BPR (Bayesian Personalized Ranking Loss)
class BPRLoss(nn.Module):
    """
    Tối ưu hóa thứ tự xếp hạng: Sản phẩm tích cực phải có điểm cao hơn sản phẩm tiêu cực
    Loss = -ln(sigmoid(score_pos - score_neg))
    """
    def __init__(self):
        super().__init__()

    def forward(self, pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
        diff = pos_scores - neg_scores
        loss = -F.logsigmoid(diff).mean()
        return loss

# Khởi tạo mô hình
model = DynamicTwoTower(feature_dim=content_vectors.shape[1], emb_dim=32).to(device)
criterion = BPRLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

# Chuyển toàn bộ Item Content Vectors sang Tensor trên Device để tra cứu nhanh
item_content_tensor = torch.tensor(content_vectors, dtype=torch.float32).to(device)

print(f"✅ Mô hình DynamicTwoTower đã sẵn sàng:")
print(model)


In [ ]:
# ===============================================================
# 6. QUÁ TRÌNH HUẤN LUYỆN (TRAINING LOOP VỚI VALIDATION)
# ===============================================================
EPOCHS = 40
history = {
    "train_loss": [],
    "test_loss": [],
    "hit_rate_at_10": [],
    "ndcg_at_10": []
}

best_test_loss = float("inf")
best_weights_path = Path("two_tower_dynamic_weights.pt")

print("🔥 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH DYNAMIC TWO-TOWER TRÊN KAGGLE...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    
    for user_vec, pos_idx, neg_idx in train_loader:
        user_vec = user_vec.to(device)
        pos_idx = pos_idx.to(device)
        neg_idx = neg_idx.to(device)
        
        pos_item_vec = item_content_tensor[pos_idx]
        neg_item_vec = item_content_tensor[neg_idx]
        
        optimizer.zero_grad()
        
        pos_scores = model(user_vec, pos_item_vec)
        neg_scores = model(user_vec, neg_item_vec)
        
        loss = criterion(pos_scores, neg_scores)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item() * len(pos_idx)
        
    scheduler.step()
    avg_train_loss = total_train_loss / len(training_samples)
    
    # Đánh giá trên tập Validation/Test
    model.eval()
    total_test_loss = 0.0
    hits_10 = 0
    ndcgs_10 = 0
    
    with torch.no_grad():
        for user_vec, pos_idx, neg_idx in test_loader:
            user_vec = user_vec.to(device)
            pos_idx = pos_idx.to(device)
            neg_idx = neg_idx.to(device)
            
            pos_item_vec = item_content_tensor[pos_idx]
            neg_item_vec = item_content_tensor[neg_idx]
            
            pos_scores = model(user_vec, pos_item_vec)
            neg_scores = model(user_vec, neg_item_vec)
            
            t_loss = criterion(pos_scores, neg_scores)
            total_test_loss += t_loss.item() * len(pos_idx)
            
            # Tính xếp hạng của pos_item so với toàn bộ 196 items trong catalog
            u_emb = model.user_embed(user_vec)
            all_i_emb = model.item_embed(item_content_tensor)
            all_scores = (u_emb @ all_i_emb.T).cpu().numpy() # shape (B, 196)
            
            for b in range(len(pos_idx)):
                target_idx = pos_idx[b].item()
                top10 = np.argsort(-all_scores[b])[:10]
                if target_idx in top10:
                    hits_10 += 1
                    rank = np.where(top10 == target_idx)[0][0] + 1
                    ndcgs_10 += 1.0 / np.log2(rank + 1)
                    
    avg_test_loss = total_test_loss / max(1, len(test_samples))
    hr_10 = hits_10 / max(1, len(test_samples))
    ndcg_10 = ndcgs_10 / max(1, len(test_samples))
    
    history["train_loss"].append(avg_train_loss)
    history["test_loss"].append(avg_test_loss)
    history["hit_rate_at_10"].append(hr_10)
    history["ndcg_at_10"].append(ndcg_10)
    
    # Lưu trọng số tốt nhất
    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        torch.save(model.state_dict(), best_weights_path)
        save_msg = "💾 (Saved Best Model)"
    else:
        save_msg = ""
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | HR@10: {hr_10:.4f} | NDCG@10: {ndcg_10:.4f} {save_msg}")

elapsed = time.time() - start_time
print(f"🎉 Huấn luyện hoàn tất trong: {elapsed:.2f} giây! Best Test Loss: {best_test_loss:.4f}")


In [ ]:
# ===============================================================
# 7. TRỰC QUAN HÓA KẾT QUẢ HUẤN LUYỆN (LOSS & METRICS PLOTS)
# ===============================================================
plt.figure(figsize=(14, 5))

# Biểu đồ Loss
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train BPR Loss", color="#2563eb", lw=2)
plt.plot(history["test_loss"], label="Test BPR Loss", color="#dc2626", lw=2, linestyle="--")
plt.title("📉 Quá trình hội tụ của hàm mất mát BPR", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

# Biểu đồ Ranking Metrics (HR@10 & NDCG@10)
plt.subplot(1, 2, 2)
plt.plot(history["hit_rate_at_10"], label="Hit Rate@10", color="#16a34a", lw=2)
plt.plot(history["ndcg_at_10"], label="NDCG@10", color="#9333ea", lw=2)
plt.title("📈 Độ chính xác xếp hạng trên tập Test", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ===============================================================
# 8. TRỰC QUAN HÓA KHÔNG GIAN LATENT BẰNG T-SNE
# ===============================================================
# Nạp lại trọng số tốt nhất để phân tích
model.load_state_dict(torch.load(best_weights_path))
model.eval()

with torch.no_grad():
    item_embs = model.item_embed(item_content_tensor).cpu().numpy()

tsne = TSNE(n_components=2, perplexity=15, random_state=42)
item_2d = tsne.fit_transform(item_embs)

tsne_df = pd.DataFrame({
    "x": item_2d[:, 0],
    "y": item_2d[:, 1],
    "category": item_df["category"].values,
    "brand": item_df["brand"].values,
    "name": item_df["name"].values
})

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=tsne_df,
    x="x", y="y",
    hue="category",
    style="brand",
    s=120,
    alpha=0.85
)
plt.title("🌌 Không gian Latent Representations của Laptop (Chiếu bởi Item Tower)", fontsize=14, fontweight="bold")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# ====================================================================
# 9. THỬ NGHIỆM TÌNH HUỐNG THỰC TẾ: BẢN DEMO DYNAMIC REAL-TIME PERSONALIZATION
# ====================================================================
print("🎯 KIỂM TRA TÍNH NĂNG BIẾN ĐỔI GU THỜI GIAN THỰC (ZERO RETRAIN):")

def get_realtime_recommendations(interacted_laptop_indices: List[Tuple[int, float]], top_k: int = 5):
    """
    Mô phỏng chính xác những gì xảy ra trên website khi khách hàng click laptop:
    1. Nhận danh sách các laptop vừa xem/thích kèm trọng số tương tác.
    2. Tính Dynamic User Profile Vector (400 chiều).
    3. Đưa qua mô hình Two-Tower -> Xuất Top-K gợi ý tức thì.
    """
    user_vec = np.zeros(content_vectors.shape[1], dtype=np.float32)
    w_sum = 0.0
    for idx, w in interacted_laptop_indices:
        user_vec += w * content_vectors[idx]
        w_sum += w
    user_vec /= max(1e-9, w_sum)
    
    # Model forward pass
    with torch.no_grad():
        u_tensor = torch.tensor(user_vec, dtype=torch.float32).unsqueeze(0).to(device)
        u_emb = model.user_embed(u_tensor)
        all_i_emb = model.item_embed(item_content_tensor)
        scores = (u_emb @ all_i_emb.T).squeeze(0).cpu().numpy()
        
    ranked_indices = np.argsort(-scores)
    
    # Loại trừ các máy đã xem
    viewed_indices = {i for i, _ in interacted_laptop_indices}
    recommendations = []
    for idx in ranked_indices:
        if idx in viewed_indices:
            continue
        recommendations.append((
            item_df.iloc[idx]["name"],
            item_df.iloc[idx]["brand"],
            item_df.iloc[idx]["category"],
            item_df.iloc[idx]["price"],
            float(scores[idx])
        ))
        if len(recommendations) >= top_k:
            break
            
    return pd.DataFrame(recommendations, columns=["Tên Laptop", "Thương hiệu", "Phân khúc", "Giá tiền", "Model Score"])

# Kịch bản 1: Khách hàng vừa click xem 2 laptop GAMING
gaming_indices = item_df[item_df["category"] == "Gaming"].index[:2].tolist()
print("\n--- 🎮 KỊCH BẢN 1: KHÁCH VỪA XEM 2 LAPTOP GAMING ---")
for idx in gaming_indices:
    print(f"👉 Vừa xem: {item_df.iloc[idx]['name']} ({item_df.iloc[idx]['brand']}) - Giá: {item_df.iloc[idx]['price']:,.0f}đ")

df_rec_gaming = get_realtime_recommendations([(gaming_indices[0], 1.0), (gaming_indices[1], 1.5)], top_k=5)
print(df_rec_gaming.to_string())

# Kịch bản 2: Khách hàng chuyển sang xem 2 laptop VĂN PHÒNG / MỎNG NHẸ
office_indices = item_df[item_df["category"].str.contains("Văn phòng|Mỏng nhẹ", case=False, na=False)].index[:2].tolist()
if not office_indices:
    office_indices = [20, 25]
print("\n--- 💼 KỊCH BẢN 2: KHÁCH CHUYỂN GU SANG XEM LAPTOP VĂN PHÒNG ---")
for idx in office_indices:
    print(f"👉 Vừa xem: {item_df.iloc[idx]['name']} ({item_df.iloc[idx]['brand']}) - Giá: {item_df.iloc[idx]['price']:,.0f}đ")

df_rec_office = get_realtime_recommendations([(office_indices[0], 1.0), (office_indices[1], 1.5)], top_k=5)
print(df_rec_office.to_string())


In [ ]:
# ====================================================================
# 10. ĐÓNG GÓI VÀ XUẤT ARTIFACTS ĐỂ TẢI VỀ TRIỂN KHAI VÀO WEB
# ====================================================================
print("📦 ĐANG ĐÓNG GÓI CÁC TỆP ARTIFACTS...")

output_dir = Path("./model_artifacts")
output_dir.mkdir(exist_ok=True)

# 1. Lưu trọng số mô hình
torch.save(model.state_dict(), output_dir / "two_tower_dynamic_weights.pt")

# 2. Lưu ma trận Content Vectors
np.save(output_dir / "content_vectors.npy", content_vectors)

# 3. Lưu ID Mappings
with open(output_dir / "id_mappings.json", "w", encoding="utf-8") as f:
    json.dump({
        "item_id_list": item_id_list,
        "content_dim": int(content_vectors.shape[1]),
        "emb_dim": 32,
        "model_type": "DynamicTwoTower",
        "trained_at": time.strftime("%Y-%m-%dT%H:%M:%SZ")
    }, f, indent=2, ensure_ascii=False)

# 4. Lưu Metrics
metrics_summary = {
    "hit_rate_at_10": float(history["hit_rate_at_10"][-1]),
    "ndcg_at_10": float(history["ndcg_at_10"][-1]),
    "best_test_loss": float(best_test_loss),
    "total_catalog_items": len(item_id_list)
}
with open(output_dir / "model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

# 5. Tạo file zip để người dùng tải về 1 click trên Kaggle
zip_filename = "model_artifacts.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in output_dir.glob("*"):
        zipf.write(file_path, arcname=file_path.name)

print(f"🎉 HOÀN TẤT XUẤT FILE THÀNH CÔNG!")
print(f"📁 Tệp nén đã tạo tại: {Path(zip_filename).resolve()}")
print(f"👉 Trên giao diện Kaggle: Hãy nhìn vào panel bên phải (Kaggle Output) -> Click nút Download file '{zip_filename}' về máy!")
